# PureVQ-GAN validation in the paper's setting — Narcissus on CIFAR-10 (1 full seed)
Runs the ladder from `repro/defense/VALIDATION.md`: **clean reference → attack works undefended → train purifier on the
poisoned set → defended**. Run the cells top to bottom. Use a **GPU runtime** (Runtime → Change runtime type; A100/L4 if you
can). Everything heavy writes to Google Drive, so a disconnect only costs you the cell that was running: just re-run the
notebook from the top (finished steps are skipped).

**Thresholds, fixed BEFORE running (edit now if you disagree, never after):**
| rung | check | pass if |
|---|---|---|
| 1 | clean reference accuracy | nat. acc >= 92 % |
| 2 | attack works undefended | PSR >= 30 % (paper: 43.95 +- 33.60) |
| 3 | purifier health on clean test | codes_used >= 200 of 512; report PSNR (no 40 dB requirement) |
| 4 | defended | PSR <= 10 % (paper: 1.64) **and** nat. acc >= 88 % (paper: 94.61) |

Honesty notes: the attack here is *our compact re-implementation* of Narcissus (not the authors' code), POOD = CIFAR-100
instead of Tiny-ImageNet, no trigger-refinement stage; rung 2 is what tells you whether it is a usable attack.

In [ ]:
import os, sys, json, subprocess, time
FAKE = bool(int(os.environ.get('PVQ_FAKE', '0')))     # 1 = tiny smoke run of the whole notebook (set only for debugging)
USE_DRIVE = not FAKE
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/pvq_validation' if USE_DRIVE else '/tmp/pvq_validation'
os.makedirs(WORK, exist_ok=True)
print('work dir:', WORK)

## 0. Get the code

In [ ]:
if not os.path.exists('/content/brainwash-repro') and not FAKE:
    !git clone -b claude/adoring-clarke-97rxkd https://github.com/anushkathagle/brainwash-repro.git /content/brainwash-repro
REPO = '/content/brainwash-repro' if not FAKE else os.getcwd()
%cd $REPO
!git log --oneline | head -3
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert FAKE or torch.cuda.is_available(), 'Switch to a GPU runtime first'

In [ ]:
# training knobs (paper values unless noted)
SEED = 0
PURIFIER_EPOCHS = 2 if FAKE else 100      # paper App. B.1
CLASSIFIER_EPOCHS = 1 if FAKE else 40     # our choice; the paper gives no classifier recipe
NS = dict(target_class=2, budget=30 if FAKE else 500,   # 500 = 1 % of CIFAR-10 train (the paper's budget)
          surrogate_epochs=1 if FAKE else 30, trigger_steps=20 if FAKE else 2000)
NPZ = f'{WORK}/ns_poisoned_s{SEED}.npz'
CLEAN_NPZ = f'{WORK}/cifar10_clean.npz'
PUR_DIR = f'{WORK}/purifier_ns_s{SEED}'
RES = f'{WORK}/results'
def run(cmd):
    print('$', cmd); t=time.time()
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:]); 
    if r.returncode: print(r.stderr[-3000:]); raise SystemExit(f'FAILED: {cmd}')
    print(f'({time.time()-t:.0f}s)'); return r.stdout
def last_json(out): return json.loads([l for l in out.splitlines() if l.startswith('{')][-1])

## 1. Smoke test (about 1 min). Must print an `[epoch]` and `[done]` line before you spend hours.

In [ ]:
run(f'python -m purevqgan.train --dataset fake --epochs 1 --batch_size 32 --base_ch 32 --out /tmp/pvq_smoke')

## 2. Build the poisoned dataset (Narcissus attack, ~30-60 min on a T4, less on an A100)
Prints surrogate training and trigger optimization progress. `surrogate-target-rate` should climb towards 1.0; if it stays near 0 the trigger did not learn and rung 2 will fail.

In [ ]:
import numpy as np
sys.path.insert(0, f'{REPO}/repro/defense')
from narcissus_lite import load_cifar, fake_cifar, build_narcissus
if os.path.exists(NPZ):
    print('poisoned set already exists, skipping:', NPZ)
else:
    data = fake_cifar() if FAKE else load_cifar('/content/data')
    out = build_narcissus(data, seed=SEED, **NS)
    np.savez(NPZ, **out)
    np.savez(CLEAN_NPZ, x_train=data['x_train'], y_train=data['y_train'])
    print('saved', NPZ)
d = np.load(NPZ, allow_pickle=True)
diff = np.abs(d['x_train'].astype(int) - d['x_train_clean'].astype(int))
print('poisoned images:', int((diff.reshape(len(diff), -1).max(1) > 0).sum()), '| max pixel change (of 255):', diff.max(),
      '| test-time trigger Linf (of 1.0):', float(np.abs(d['trigger']).max()))

## 3. Rung 1 — clean reference (no attack, no defense)
The ResNet-18 recipe is ours; the pass bar is just that the recipe is sane (>= 92 %).

In [ ]:
o = run(f'python repro/defense/validate_paper.py --stage classify --poison_npz {NPZ} --clean_train_npz {CLEAN_NPZ} '
        f'--epochs {CLASSIFIER_EPOCHS} --seed {SEED} --tag clean_ref --out {RES}')
clean = last_json(o); print('RUNG 1:', 'PASS' if clean['nat_acc'] >= 92 else 'FAIL (or FAKE run)', clean['nat_acc'])

## 4. Rung 2 — the attack must work undefended

In [ ]:
o = run(f'python repro/defense/validate_paper.py --stage classify --poison_npz {NPZ} '
        f'--epochs {CLASSIFIER_EPOCHS} --seed {SEED} --tag undefended --out {RES}')
undef = last_json(o); print('RUNG 2:', 'PASS' if undef['psr'] >= 30 else 'FAIL (or FAKE run)', 'PSR', undef['psr'], 'acc', undef['nat_acc'])
# If this FAILS: do not continue. Raise NS['budget'] or trigger_steps / surrogate_epochs, delete NPZ, and rebuild (cell 2).

## 5. Train the PureVQ-GAN purifier on the POISONED training set (paper Sec. 3.3 protocol)
About 1.5-2 h on a T4, ~30-40 min on an A100. **Resumable**: if Colab disconnects, re-run this cell. Check `codes_used` and
`val_psnr` in the printed epochs; it should stay in the hundreds of codes and PSNR should rise.

In [ ]:
def stream(cmd):
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout: print(line, end='', flush=True)
    if p.wait(): raise SystemExit(f'FAILED: {cmd}')
if os.path.exists(f'{PUR_DIR}/purifier.pt'):
    print('purifier already trained')
else:
    small = '--base_ch 32 --batch_size 64' if FAKE else ''
    stream(f'python -m purevqgan.train --dataset npy:{NPZ} --img_size 32 --latent_size 8 --num_codes 512 --amp --resume '
           f'--epochs {PURIFIER_EPOCHS} --seed {SEED} --out {PUR_DIR} {small}')

## 6. Rung 3 — purifier health on clean test images

In [ ]:
o = run(f'python repro/defense/validate_paper.py --stage purifier --poison_npz {NPZ} --purifier {PUR_DIR}/purifier.pt --seed {SEED} --tag pvq --out {RES}')
h = last_json(o); print('RUNG 3:', 'PASS' if h['codes_used'] >= 200 else 'FAIL (or FAKE run)', 'codes', h['codes_used'], 'PSNR', round(h['psnr_clean_test'],2))
print('perturbation survival (poisoned train rows):', h.get('perturbation'))

## 7. Rung 4 — defended (two variants, since the paper is ambiguous)
(a) classifier trained on purified training data only; (b) additionally purify the test inputs, with the trigger added **before** purification.

In [ ]:
res = {}
for tag, extra in [('pvq_train_only', ''), ('pvq_train_and_test', '--purify_test')]:
    o = run(f'python repro/defense/validate_paper.py --stage classify --poison_npz {NPZ} --purifier {PUR_DIR}/purifier.pt {extra} '
            f'--epochs {CLASSIFIER_EPOCHS} --seed {SEED} --tag {tag} --out {RES}')
    res[tag] = last_json(o)
for tag, r in res.items():
    ok = r['psr'] <= 10 and r['nat_acc'] >= 88
    print(f"RUNG 4 [{tag}]:", 'PASS' if ok else 'FAIL (or FAKE run)', '| PSR', round(r['psr'],2), '| acc', round(r['nat_acc'],2))

## 8. Summary (copy this into your notes)

In [ ]:
rows = [('clean reference (no attack)', clean), ('undefended', undef)] + list(res.items())
print(f"{'condition':32} {'nat acc %':>10} {'PSR %':>8}   paper (PSR / acc)")
paper = {'undefended': '43.95 / 94.89', 'pvq_train_only': '1.64 / 94.61', 'pvq_train_and_test': '1.64 / 94.61'}
for name, r in rows:
    print(f"{name:32} {r['nat_acc']:10.2f} {r.get('psr', float('nan')):8.2f}   {paper.get(name, '-')}")
print('\npurifier: codes_used', h['codes_used'], '| clean-test PSNR', round(h['psnr_clean_test'], 2), 'dB | params', round(h['params_M'], 1), 'M')
print('Single seed: PSR for Narcissus varies a lot between runs (paper std 33.6), so treat this as a sanity check, not a replication of the mean.')

## If something fails
* **Rung 2 fails** (attack too weak): engineering problem, not evidence about PureVQ-GAN. Fix the attack first.
* **Rung 3 fails** (codebook collapse / bad PSNR): paste the `train_log.jsonl` lines to me.
* **Rung 4 fails but 1-3 pass**: follow the order in `VALIDATION.md` (bigger purifier `--base_ch 256 --num_res_blocks 2`, longer training, two passes). Record each attempt; a persistent failure is a reportable finding.